# Biomarker ranking and validation from cancer cell-line transcriptomics
**Reference:** **Tibor Fekete, J., & Győrffy, B. (2022). A unified platform enabling biomarker ranking and validation for 1562 drugs using transcriptomic data of 1250 cancer cell lines.** _Computational and Structural Biotechnology Journal_, 20, **2885–2894**. https://doi.org/10.1016/j.csbj.2022.06.007
 @tiborfeketeUnifiedPlatformEnabling2022

This notebook implements the workflow described by Fekete and Győrffy (2022):
1. Normalize expression data.
2. Label the lower response tertile as **sensitive** and the upper tertile as **resistant**; omit the middle tertile.
3. Randomly split samples into 66% training and 34% testing sets.
4. Select candidate genes with a training-set Mann-Whitney U test at $p<0.05$.
5. Combine selected genes in a random-forest classifier.
6. Evaluate the untouched test set using ROC AUC, accuracy, Cohen's kappa, sensitivity, specificity, and precision.

The implementation also includes single-gene ROC/Mann-Whitney results, Spearman correlations, Benjamini-Hochberg FDR, cell-line ranking, and a correlation matrix.

> **Important reproducibility note:** the article does not report a random seed or all random-forest hyperparameters. The workflow can be reproduced, but exact values in its Table 4 are not guaranteed without the authors' precise split and model settings.

## Key corrections to the original draft

- Feature selection is performed **after the train/test split and only on the training set**, preventing information leakage.
- The CCLE normalization includes zero-expression filtering, DESeq-style median-of-ratios normalization, quantile normalization, and per-cell-line scaling to a mean of 1000.
- GDSC RMA expression is treated as already background-corrected and quantile-normalized, then scaled to a mean of 1000 per cell line.

In [1]:
import pandas as pd
import numpy as np
import qnorm
import warnings
from scipy.stats import mannwhitneyu
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, precision_score

warnings.filterwarnings('ignore')

## 1. Load and normalize expression data

The article used two expression pipelines:

### 1.1 CCLE/DepMap/CTRP RNA-seq
- **Raw Data:** `CCLE_RNAseq_reads.csv`
- **Source:** https://figshare.com/articles/dataset/public_21q1/13681534


In [2]:
# Data import
counts = pd.read_csv(
    "../data/raw_data/expression/CCLE_RNAseq_reads.csv",
    index_col=0
)

In [3]:
counts.iloc[:5, :5]

,TSPAN6 (ENSG00000000003),TNMD (ENSG00000000005),DPM1 (ENSG00000000419),SCYL3 (ENSG00000000457),C1orf112 (ENSG00000000460)
ACH-001113,2383.0,0.0,4766.0,961.74,1518.30
ACH-001289,2529.0,13.0,3857.0,555.00,833.00
ACH-001339,1552.0,0.0,8057.0,849.46,2511.50
ACH-001538,5657.0,0.0,6056.0,1056.70,656.26
ACH-000242,22806.0,0.0,4702.0,1109.30,1499.70


1. DESeq normalization

In [4]:
# DESeq normalization
from pydeseq2.preprocessing import deseq2_norm

norm_counts, size_factors = deseq2_norm(counts)

In [5]:
# Quantile normalization
import pandas as pd
import qnorm

norm_counts = pd.DataFrame(
    norm_counts,
    index=counts.index,
    columns=counts.columns
)

norm_quantile = qnorm.quantile_normalize(
    norm_counts,
    axis=1
)

2. Scale each cell line to mean expression 1000.

In [6]:
# Scale each cell line to mean expression 1000.
expression_scaled = norm_quantile.div(
    norm_quantile.mean(axis=1),
    axis=0
) * 1000

In [7]:
row_means = expression_scaled.mean(axis=1)

print(row_means.describe())
print("Minimum:", row_means.min())
print("Maximum:", row_means.max())

count    1.376000e+03
mean     1.000000e+03
std      3.958246e-11
min      1.000000e+03
25%      1.000000e+03
50%      1.000000e+03
75%      1.000000e+03
max      1.000000e+03
dtype: float64
Minimum: 999.9999999998709
Maximum: 1000.0000000001177


3. Remove genes with zero counts in more than half of cell lines

In [8]:
# Remove genes with zero counts in more than half of cell lines
# Determine zero prevalence using DESeq-normalized counts
zero_fraction = (norm_counts == 0).mean(axis=0)

keep_genes = zero_fraction <= 0.5

print("Genes before filtering:", norm_counts.shape[1])
print("Genes retained:", keep_genes.sum())
print("Genes removed:", (~keep_genes).sum())

expression_final = expression_scaled.loc[:, keep_genes].copy()
print("Final expression shape:", expression_final.shape)

Genes before filtering: 52405
Genes retained: 25772
Genes removed: 26633
Final expression shape: (1376, 25772)


4. Resolving duplicate labels for genes

In [9]:
gene_annotation = pd.DataFrame({
    "original_name": expression_final.columns,
    "gene_symbol": expression_final.columns.str.extract(
        r"^(.+?)\s+\(ENSG", expand=False
    ),
    "ensembl_id": expression_final.columns.str.extract(
        r"\((ENSG\d+)\)", expand=False
    )
})

gene_annotation.head()

,original_name,gene_symbol,ensembl_id
0,TSPAN6 (ENSG00000000003),TSPAN6,ENSG00000000003
1,DPM1 (ENSG00000000419),DPM1,ENSG00000000419
2,SCYL3 (ENSG00000000457),SCYL3,ENSG00000000457
3,C1orf112 (ENSG00000000460),C1orf112,ENSG00000000460
4,FGR (ENSG00000000938),FGR,ENSG00000000938


In [10]:
symbols = gene_annotation["gene_symbol"].fillna(
    gene_annotation["original_name"]
)

print("Duplicated symbols:", symbols.duplicated().sum())

Duplicated symbols: 3


In [11]:
dup_mask = symbols.duplicated(keep=False)

duplicates = gene_annotation.loc[dup_mask].copy()
duplicates["analysis_symbol"] = symbols[dup_mask].values

duplicates.sort_values("analysis_symbol")

,original_name,gene_symbol,ensembl_id,analysis_symbol
11165,POLR2J3 (ENSG00000168255),POLR2J3,ENSG00000168255,POLR2J3
25629,POLR2J3 (ENSG00000285437),POLR2J3,ENSG00000285437,POLR2J3
16338,POLR2J4 (ENSG00000214783),POLR2J4,ENSG00000214783,POLR2J4
23699,POLR2J4 (ENSG00000272655),POLR2J4,ENSG00000272655,POLR2J4
23131,RMRP (ENSG00000269900),RMRP,ENSG00000269900,RMRP
24494,RMRP (ENSG00000277027),RMRP,ENSG00000277027,RMRP


In [12]:
duplicate_symbols = ["POLR2J3", "POLR2J4", "RMRP"]

selected_columns = []

for symbol in duplicate_symbols:
    matching = [
        col for col in expression_final.columns
        if col.startswith(f"{symbol} (")
    ]

    means = expression_final[matching].mean(axis=0)
    selected = means.idxmax()

    print(f"{symbol}: keeping {selected}")
    print(means.sort_values(ascending=False), "\n")

    selected_columns.append(selected)

POLR2J3: keeping POLR2J3 (ENSG00000168255)
POLR2J3 (ENSG00000168255)    997.666264
POLR2J3 (ENSG00000285437)    983.097248
dtype: float64 

POLR2J4: keeping POLR2J4 (ENSG00000272655)
POLR2J4 (ENSG00000272655)    1000.687048
POLR2J4 (ENSG00000214783)     974.258522
dtype: float64 

RMRP: keeping RMRP (ENSG00000277027)
RMRP (ENSG00000277027)    1015.725225
RMRP (ENSG00000269900)    1009.458878
dtype: float64 



In [13]:
columns_to_remove = []

for symbol, selected in zip(duplicate_symbols, selected_columns):
    matching = [
        col for col in expression_final.columns
        if col.startswith(f"{symbol} (")
    ]

    columns_to_remove.extend(
        col for col in matching if col != selected
    )

expression_unique = expression_final.drop(columns=columns_to_remove)

In [14]:
expression_unique.columns = expression_unique.columns.str.replace(
    r"\s+\(ENSG\d+\)$",
    "",
    regex=True
)

print("Duplicated symbols:",
      expression_unique.columns.duplicated().sum())

Duplicated symbols: 0


In [15]:
duplicate_resolution = pd.DataFrame({
    "symbol": duplicate_symbols,
    "kept_column": selected_columns
})

duplicate_resolution

,symbol,kept_column
0,POLR2J3,POLR2J3 (ENSG00000168255)
1,POLR2J4,POLR2J4 (ENSG00000272655)
2,RMRP,RMRP (ENSG00000277027)


In [16]:
expression_final = expression_unique.div(
    expression_unique.mean(axis=1),
    axis=0
) * 1000

## 1.2 **GDSC1/GDSC2 RNA-normalized Affymetrix HGU-219 matrix:**
- As of 19/7/2026, the data is not available on the original website anymore.
- This source was used to trace back the original link: https://github.com/BorgwardtLab/Kernelized-Rank-Learning/blob/master/URLs_to_GDSC_datasets.txt
- **Raw Data:** `Cell_line_RMA_proc_basalExp.txt`
- **Original Source:** http://www.cancerrxgene.org/gdsc1000/GDSC1000_WebResources/Data/preprocessed/Cell_line_RMA_proc_basalExp.txt.zip
- **Currently Available Source:** https://web.archive.org/web/20260209143748/http://www.cancerrxgene.org/gdsc1000/GDSC1000_WebResources/Data/preprocessed/Cell_line_RMA_proc_basalExp.txt.zip

1. Start from RMA-normalized HGU-219 expression.
2. Apply the per-cell-line scaling to mean 1000.

In [17]:
# Data import
import pandas as pd
import numpy as np

gdsc_raw = pd.read_csv(
    "../data/raw_data/expression/Cell_line_RMA_proc_basalExp.txt",
    sep="\t",
    low_memory=False
)

In [18]:
gdsc_raw.columns[:10].tolist()

['GENE_SYMBOLS',
 'GENE_title',
 'DATA.906826',
 'DATA.687983',
 'DATA.910927',
 'DATA.1240138',
 'DATA.1240139',
 'DATA.906792',
 'DATA.910688',
 'DATA.1240135']

In [19]:
gdsc_raw.iloc[:5, :5]

,GENE_SYMBOLS,GENE_title,DATA.906826,DATA.687983,DATA.910927
0,TSPAN6,tetraspanin 6 [Source:HGNC Symbol;Acc:11858],7.632023,7.548671,8.712338
1,TNMD,tenomodulin [Source:HGNC Symbol;Acc:17757],2.964585,2.777716,2.643508
2,DPM1,dolichyl-phosphate mannosyltransferase polypep...,10.379553,11.807341,9.880733
3,SCYL3,SCY1-like 3 (S. cerevisiae) [Source:HGNC Symbo...,3.614794,4.066887,3.956230
4,C1orf112,chromosome 1 open reading frame 112 [Source:HG...,3.380681,3.732485,3.236620


In [20]:
gdsc_raw.shape

(17737, 1020)

In [21]:
# Identify symbol and title columns
symbol_col = "GENE_SYMBOLS"
title_col = "GENE_title"

In [22]:
# Construct gene-by-cell-line matrix
expression_cols = [
    col for col in gdsc_raw.columns
    if col not in {symbol_col, title_col}
]

gdsc_gene_by_sample = (
    gdsc_raw[expression_cols]
    .astype("float64")
    .copy()
)

gdsc_gene_by_sample.index = (
    gdsc_raw[symbol_col]
    .astype("string")
    .str.strip()
)

In [23]:
gdsc_gene_by_sample.shape

(17737, 1018)

In [24]:
gdsc_gene_by_sample.iloc[:5, :5]

,DATA.906826,DATA.687983,DATA.910927,DATA.1240138,DATA.1240139
GENE_SYMBOLS,,,,,
TSPAN6,7.632023,7.548671,8.712338,7.797142,7.729268
TNMD,2.964585,2.777716,2.643508,2.817923,2.957739
DPM1,10.379553,11.807341,9.880733,9.883471,10.418840
SCYL3,3.614794,4.066887,3.956230,4.063701,4.341500
C1orf112,3.380681,3.732485,3.236620,3.558414,3.840373


2. Identify unambiguous gene names, remove rows with missing symbols
- **Source:** https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt

In [77]:
symbols_raw = (
    gdsc_raw["GENE_SYMBOLS"]
    .astype("string")
    .str.strip()
)

In [78]:
print("Missing values:", symbols_raw.isna().sum())
print("Blank strings:", symbols_raw.eq("").sum())

Missing values: 318
Blank strings: 0


In [79]:
gdsc_missing = (
    gdsc_raw.loc[
    symbols_raw.isna(),
    ["GENE_SYMBOLS", "GENE_title"]
    ]
)

In [80]:
gdsc_missing

,GENE_SYMBOLS,GENE_title
82,NaN,Putative RNA exonuclease NEF-sp [Source:UniPr...
1187,NaN,selenoprotein O [Source:RefSeq peptide;Acc:NP...
1748,NaN,ProSAP-interacting protein 1 [Source:UniProtK...
1870,NaN,Mitogen-activated protein kinase kinase kinase...
3006,NaN,Lipid phosphate phosphatase-related protein ty...
...,...,...
17725,NaN,NaN
17729,NaN,NaN
17730,NaN,NaN
17732,NaN,hsa-mir-5195 [Source:miRBase;Acc:MI0018174]


In [81]:
print(
    "Missing-symbol rows with a title:",
    gdsc_missing["GENE_title"].notna().sum()
)

print(
    "Unique titles:",
    gdsc_missing["GENE_title"].nunique(dropna=True)
)

Missing-symbol rows with a title: 68
Unique titles: 68


In [82]:
gdsc_missing["GENE_title"].value_counts(dropna=False).head(30)

GENE_title
NaN                                                                                                                         250
Putative RNA exonuclease NEF-sp  [Source:UniProtKB/Swiss-Prot;Acc:Q96IC2]                                                     1
Uncharacterized protein  [Source:UniProtKB/TrEMBL;Acc:C9J7I0]                                                                 1
Uncharacterized protein  [Source:UniProtKB/TrEMBL;Acc:E9PE83]                                                                 1
Uncharacterized protein; cDNA FLJ53535  [Source:UniProtKB/TrEMBL;Acc:B4DK98]                                                  1
Apolipoprotein C-II  [Source:UniProtKB/Swiss-Prot;Acc:P02655]                                                                 1
HCG20425, isoform CRA_a; Uncharacterized protein; cDNA FLJ53815  [Source:UniProtKB/TrEMBL;Acc:B4DNA4]                         1
Protein LOC100130015; cDNA FLJ52651  [Source:UniProtKB/TrEMBL;Acc:B7Z8Y8]                    

In [105]:
# Extract source database and accession
gdsc_missing["source_database"] = (
    gdsc_missing["GENE_title"]
    .str.extract(
        r"\[Source:([^;\]]+);Acc:[^\]]+\]",
        expand=False
    )
)

gdsc_missing["source_accession"] = (
    gdsc_missing["GENE_title"]
    .str.extract(
        r"\[Source:[^;\]]+;Acc:([^\]]+)\]",
        expand=False
    )
)
gdsc_annotation = gdsc_missing.loc[
    gdsc_missing["source_accession"].notna()

]
gdsc_annotation["source_accession"] = (
    gdsc_annotation["source_accession"]
    .astype("string")
    .str.strip()
)

gdsc_annotation["source_database"] = (
    gdsc_annotation["source_database"]
    .astype("string")
    .str.strip()
)

In [106]:
gdsc_annotation

,GENE_SYMBOLS,GENE_title,source_database,source_accession
82,NaN,Putative RNA exonuclease NEF-sp [Source:UniPr...,UniProtKB/Swiss-Prot,Q96IC2
1187,NaN,selenoprotein O [Source:RefSeq peptide;Acc:NP...,RefSeq peptide,NP_113642
1748,NaN,ProSAP-interacting protein 1 [Source:UniProtK...,UniProtKB/Swiss-Prot,O60299
1870,NaN,Mitogen-activated protein kinase kinase kinase...,UniProtKB/Swiss-Prot,Q9NYL2
3006,NaN,Lipid phosphate phosphatase-related protein ty...,UniProtKB/Swiss-Prot,Q96GM1
...,...,...,...,...
17616,NaN,Protocadherin-psi1; Uncharacterized protein [...,UniProtKB/TrEMBL,Q96T98
17620,NaN,"HCG2014417, isoform CRA_a; Uncharacterized pro...",UniProtKB/TrEMBL,B7Z718
17688,NaN,Uncharacterized protein [Source:UniProtKB/TrE...,UniProtKB/TrEMBL,H0YMM6
17722,NaN,Uncharacterized protein [Source:UniProtKB/TrE...,UniProtKB/TrEMBL,J3KSW9


In [99]:
gdsc_annotation["source_database"].value_counts(dropna=False)

source_database
UniProtKB/TrEMBL        30
UniProtKB/Swiss-Prot    29
RefSeq peptide           7
miRBase                  2
Name: count, dtype: int64

In [101]:
# Annotation recovery table
import pandas as pd
import numpy as np

hgnc = pd.read_csv(
    "../data/raw_data/annotation/hgnc_complete_set.txt",
    sep="\t",
    dtype="string",
    low_memory=False
)

In [102]:
hgnc.shape

(45021, 54)

In [110]:
hgnc.head()

,hgnc_id,symbol,name,locus_group,locus_type,status,location,location_sortable,alias_symbol,alias_name,...,cd,lncrnadb,enzyme_id,intermediate_filament_db,rna_central_id,lncipedia,gtrnadb,agr,mane_select,gencc
0,HGNC:5,A1BG,alpha-1-B glycoprotein,protein-coding gene,gene with protein product,Approved,19q13.43,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,HGNC:5,ENST00000263100.8|NM_130786.4,<NA>
1,HGNC:37133,A1BG-AS1,A1BG antisense RNA 1,non-coding RNA,"RNA, long non-coding",Approved,19q13.43,<NA>,FLJ23569,<NA>,...,<NA>,<NA>,<NA>,<NA>,URS00007E4F6E,A1BG-AS1,<NA>,HGNC:37133,<NA>,<NA>
2,HGNC:24086,A1CF,APOBEC1 complementation factor,protein-coding gene,gene with protein product,Approved,10q11.23,<NA>,ACF|ASP|ACF64|ACF65|APOBEC1CF,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,HGNC:24086,ENST00000373997.8|NM_014576.4,<NA>
3,HGNC:7,A2M,alpha-2-macroglobulin,protein-coding gene,gene with protein product,Approved,12p13.31,<NA>,FWP007|S863-7|CPAMD5,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,HGNC:7,ENST00000318602.12|NM_000014.6,HGNC:7
4,HGNC:27057,A2M-AS1,A2M antisense RNA 1,non-coding RNA,"RNA, long non-coding",Approved,12p13.31,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,URS00001F234A,A2M-AS1,<NA>,HGNC:27057,<NA>,<NA>
